# 22 - LSTM secuencial vs Ridge-ARX, N-BEATS, ARIMA y SARIMAX

Este notebook entrena una red LSTM para predecir el nivel del Mburicao a 10, 20 y 30 minutos.

El protocolo replica las condiciones principales usadas en el experimento final de N-BEATS:

- segmentos 2021 y 2025 tratados por separado;
- frecuencia regular de 10 minutos;
- split temporal 70% train, 15% validation, 15% test;
- ventana historica seleccionada entre 72, 144 y 288 pasos;
- horizonte de 3 pasos;
- grid search de hiperparametros usando validation;
- evaluacion secuencial con `stride=1` sobre test;
- metricas globales, por segmento, por lead time y por condiciones criticas.

Las tablas finales se guardan en `reports/22_lstm_vs_modelos` y se combinan con las tablas finales de `reports/15_finales` y `reports/17_arima_sarimax_baselines`.

## Guia conceptual: que es una LSTM y por que usarla aqui

Una LSTM (*Long Short-Term Memory*) es una red neuronal recurrente disenada para aprender dependencias temporales en secuencias. A diferencia de una red densa clasica, que recibe un vector fijo de variables y no tiene una nocion interna de orden temporal, una LSTM procesa una ventana como una sucesion ordenada:

```text
t-710 min, t-700 min, ..., t-20 min, t-10 min
```

En cada paso temporal, la LSTM actualiza dos estados internos:

- `hidden state`: representacion compacta de lo que el modelo considera relevante hasta ese instante;
- `cell state`: memoria de mas largo plazo, controlada por compuertas (*gates*) que deciden que informacion conservar, olvidar o incorporar.

Estas compuertas son:

- `forget gate`: decide que parte de la memoria anterior se descarta;
- `input gate`: decide que informacion nueva entra a la memoria;
- `output gate`: decide que parte de la memoria se expone como representacion para predecir.

En un problema hidrologico como el Mburicao, esta estructura es util porque el nivel actual no depende solo del ultimo valor observado. Tambien puede depender de:

- la trayectoria reciente del nivel;
- la lluvia acumulada en periodos previos;
- la velocidad de ascenso o descenso;
- patrones de respuesta retardada entre precipitacion y nivel;
- diferencias entre periodos tranquilos y eventos de crecida.

La LSTM intenta aprender estas relaciones directamente a partir de ventanas temporales, sin que tengamos que definir manualmente todas las variables ARX como `delta_10m`, `lluvia_60m`, medias moviles o drift. Sin embargo, esto no significa que sea magicamente superior: necesita buen escalado, separacion temporal estricta, validacion honesta y comparacion con modelos mas simples.

## Como leer este experimento

Este notebook separa el flujo en cinco etapas:

1. **Preparacion de datos**: se cargan las series de nivel y precipitacion, se regularizan a 10 minutos y se separan los segmentos 2021 y 2025.
2. **Construccion de ventanas**: cada ejemplo de entrenamiento es una ventana historica de nivel y lluvia. El objetivo es el nivel futuro a 10, 20 y 30 minutos.
3. **Grid search**: se prueban varias arquitecturas LSTM y longitudes de ventana. La seleccion se hace solo con validation.
4. **Entrenamiento final**: se reentrena una LSTM con la mejor configuracion seleccionada.
5. **Evaluacion en test**: se calculan metricas finales y tablas comparativas contra Ridge-ARX, N-BEATS, ARIMA y SARIMAX.

La idea metodologica importante es esta: **validation se usa para elegir hiperparametros; test se reserva para reportar resultados finales**. Esto evita elegir el modelo mirando el conjunto de prueba.

## Imports

In [ ]:
import os
import copy
import gc
import random
import time
import warnings
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

try:
    torch.set_float32_matmul_precision("medium")
except Exception as exc:
    print(f"No se pudo configurar torch.set_float32_matmul_precision: {exc}")

## Configuracion experimental

In [ ]:
TIME_COL = "Time"
TARGET_COL = "Nivel"
RAIN_COL = "Precipitacion"

FREQ = "10min"
FREQ_DELTA = pd.Timedelta(minutes=10)
STEPS_PER_HOUR = 6

FORECAST_HORIZON = 3
EVAL_STRIDE = 1

TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 1.0 - TRAIN_FRACTION - VAL_FRACTION

# Parametros alineados con el experimento final de N-BEATS.
LSTM_INPUT_LENGTH = 72
LSTM_EPOCHS = 30
LSTM_BATCH_SIZE = 1024
RANDOM_STATE = 42

# Parametros propios de la arquitectura LSTM.
LSTM_HIDDEN_SIZE = 128
LSTM_NUM_LAYERS = 2
LSTM_DROPOUT = 0.05
LSTM_LEARNING_RATE = 1e-3
LSTM_WEIGHT_DECAY = 0.0
LSTM_MODEL_NAME = "22_lstm_vs_modelos_in72_h3"

# Grid search LSTM.
# Se mantiene FORECAST_HORIZON=3 para que la comparacion final siga siendo t+10, t+20 y t+30.
RUN_LSTM_GRID_SEARCH = True
LSTM_GRID = []
for input_length, hidden_size, num_layers, dropout, learning_rate, n_epochs, batch_size in product(
    [72, 144, 288],     # 12 h, 24 h, 48 h
    [64, 128],
    [1, 2],
    [0.05],
    [1e-3],
    [30],
    [1024],
):
    LSTM_GRID.append({
        "input_length": input_length,
        "forecast_horizon": FORECAST_HORIZON,
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "weight_decay": LSTM_WEIGHT_DECAY,
        "n_epochs": n_epochs,
        "batch_size": batch_size,
    })

print(f"Cantidad de corridas LSTM planificadas: {len(LSTM_GRID)}")
display(pd.DataFrame(LSTM_GRID))

COMPARE_MODELS = [
    "Ridge_ARX_local",
    "NBEATS",
    "ARIMA_selected",
    "SARIMAX_lagged_exog_selected",
    "LSTM",
]

print("Split:")
print(f"Train: {TRAIN_FRACTION:.0%}")
print(f"Validation: {VAL_FRACTION:.0%}")
print(f"Test: {TEST_FRACTION:.0%}")

print("\nLead times evaluados:")
for step in range(1, FORECAST_HORIZON + 1):
    print(f"lead_step={step}: t+{step * 10} min")

print("\nParametros LSTM:")
print("input_length:", LSTM_INPUT_LENGTH)
print("epochs:", LSTM_EPOCHS)
print("batch_size:", LSTM_BATCH_SIZE)
print("hidden_size:", LSTM_HIDDEN_SIZE)
print("num_layers:", LSTM_NUM_LAYERS)
print("dropout:", LSTM_DROPOUT)

## Decisiones metodologicas clave

### Se entrena un modelo por segmento?

No. En este notebook se entrena **una unica LSTM global** usando ejemplos construidos a partir de ambos segmentos: 2021 y 2025. Esto significa que:

- no se concatena 2021 con 2025 como si fueran una sola serie continua;
- no se crean ventanas que crucen el salto temporal entre 2021/2022 y 2025/2026;
- cada segmento produce sus propias ventanas validas;
- las ventanas de 2021 y 2025 se juntan en un mismo arreglo de entrenamiento;
- el modelo aprende parametros compartidos a partir de ambos segmentos.

La razon es metodologica: los dos segmentos representan realizaciones del mismo sistema hidrologico, pero estan separados por un hueco temporal grande. Si los unieramos como una unica serie, la LSTM podria interpretar falsamente que el final de 2021/2022 esta seguido inmediatamente por el inicio de 2025/2026. En cambio, al construir ventanas por segmento, evitamos esa continuidad artificial.

### Se usa la lluvia como variable exogena?

Si. La precipitacion se incluye como segunda variable de entrada en cada paso de la ventana historica. En terminos de series temporales, la lluvia actua como una covariable exogena observada en el pasado reciente.

Importante: este notebook usa **lluvia historica dentro de la ventana**, no lluvia futura. Para emitir una prediccion en un origen temporal `t`, el modelo recibe informacion hasta `t` o, de forma equivalente en el codigo, hasta el ultimo instante observado antes de los valores futuros. No se le entrega la precipitacion futura en `t+10`, `t+20` o `t+30`. Esto evita fuga de informacion.

### Que aprende entonces el modelo?

La LSTM recibe secuencias como:

```text
[(nivel, lluvia) en t-720 min,
 (nivel, lluvia) en t-710 min,
 ...
 (nivel, lluvia) en t-10 min]
```

y aprende a producir:

```text
[nivel en t+10 min, nivel en t+20 min, nivel en t+30 min]
```

Por eso el modelo es multivariado en la entrada y multi-horizonte en la salida.

## Rutas del proyecto

In [ ]:
def resolve_project_dir():
    cwd = os.getcwd()
    candidates = [
        cwd,
        os.path.dirname(cwd),
        os.path.join(cwd, "Mburicao_Iberamia"),
        os.path.join(os.path.dirname(cwd), "Mburicao_Iberamia"),
        r"G:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia",
        r"D:\Google Drive\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia",
        "/data/students/federico.moran",
        "/data/students/federico.moran/notebooks",
    ]

    for candidate in candidates:
        if not candidate:
            continue
        if os.path.exists(os.path.join(candidate, "notebooks")) or os.path.exists(os.path.join(candidate, "data")):
            return os.path.abspath(candidate)

    return os.path.abspath(cwd)


PROJECT_DIR = resolve_project_dir()

FIGURES_DIR = os.path.join(PROJECT_DIR, "figures", "22_lstm_vs_modelos")
MODELS_DIR = os.path.join(PROJECT_DIR, "models", "22_lstm_vs_modelos")
REPORTS_DIR = os.path.join(PROJECT_DIR, "reports", "22_lstm_vs_modelos")
REPORTS_ROOT = os.path.join(PROJECT_DIR, "reports")

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)


def find_csv(filename):
    candidates = [
        os.path.join(PROJECT_DIR, "data", "processed", "datos_lluvia_nivel", filename),
        os.path.join(PROJECT_DIR, "datasets", "datos_lluvia_nivel", filename),
        os.path.join(os.path.dirname(PROJECT_DIR), "datasets", "datos_lluvia_nivel", filename),
        os.path.join("/data/students/federico.moran", "datasets", "datos_lluvia_nivel", filename),
        os.path.join("/data/students/federico.moran", "notebooks", "datasets", "datos_lluvia_nivel", filename),
    ]

    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate

    raise FileNotFoundError("No se encontro el archivo: " + filename + "\nCandidatos:\n" + "\n".join(candidates))


CSV_PATHS = {
    "2021": find_csv("Mburicao_2021.csv"),
    "2025": find_csv("Mburicao_2025.csv"),
}

print("PROJECT_DIR:", PROJECT_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("REPORTS_DIR:", REPORTS_DIR)
print("CSV encontrados:")
for segment_name, path in CSV_PATHS.items():
    print(segment_name, "->", path)

## Carga y regularizacion de datos

In [ ]:
def load_segment(csv_path, segment_name):
    df = pd.read_csv(csv_path)

    required_columns = {TIME_COL, TARGET_COL, RAIN_COL}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Faltan columnas requeridas en {csv_path}: {sorted(missing_columns)}")

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL])
    df = df.sort_values(TIME_COL)
    df = df.set_index(TIME_COL)
    df = df[~df.index.duplicated(keep="first")]

    df = df[[TARGET_COL, RAIN_COL]].copy()
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df[RAIN_COL] = pd.to_numeric(df[RAIN_COL], errors="coerce")

    df = df.asfreq(FREQ)
    df[TARGET_COL] = df[TARGET_COL].interpolate(method="time").ffill().bfill()
    df[RAIN_COL] = df[RAIN_COL].fillna(0.0)
    df["segmento"] = segment_name

    return df


segments = {}
for segment_name, csv_path in CSV_PATHS.items():
    segments[segment_name] = load_segment(csv_path, segment_name)

segment_names = list(segments.keys())

summary_rows = []
for segment_name, df_segment in segments.items():
    summary_rows.append({
        "segmento": segment_name,
        "filas": len(df_segment),
        "inicio": df_segment.index.min(),
        "fin": df_segment.index.max(),
        "freq_inferida": pd.infer_freq(df_segment.index),
        "nivel_min": df_segment[TARGET_COL].min(),
        "nivel_max": df_segment[TARGET_COL].max(),
        "lluvia_total": df_segment[RAIN_COL].sum(),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_path = os.path.join(REPORTS_DIR, "22_dataset_resumen.csv")
summary_df.to_csv(summary_path, index=False)
print("Resumen guardado en:", summary_path)

## Split train-validation-test

In [ ]:
split_info = {}

for segment_name in segment_names:
    df_segment = segments[segment_name]
    n = len(df_segment)

    train_end_position = int(n * TRAIN_FRACTION)
    val_end_position = int(n * (TRAIN_FRACTION + VAL_FRACTION))

    max_grid_input_length = max([LSTM_INPUT_LENGTH] + [int(params["input_length"]) for params in LSTM_GRID])
    min_history = max(max_grid_input_length, FORECAST_HORIZON + 1)

    train_end_position = max(train_end_position, min_history + FORECAST_HORIZON + 1)
    val_end_position = max(val_end_position, train_end_position + FORECAST_HORIZON + 1)
    val_end_position = min(val_end_position, n - FORECAST_HORIZON - 1)

    train_end_time = df_segment.index[train_end_position]
    test_start_time = df_segment.index[val_end_position]

    split_info[segment_name] = {
        "train_end_position": train_end_position,
        "val_end_position": val_end_position,
        "test_start_position": val_end_position,
        "train_end_time": train_end_time,
        "test_start_time": test_start_time,
        "train_points": train_end_position,
        "val_points": val_end_position - train_end_position,
        "test_points": n - val_end_position,
    }

split_df = pd.DataFrame([
    {
        "segmento": segment_name,
        "train_end_position": info["train_end_position"],
        "val_end_position": info["val_end_position"],
        "train_end_time": info["train_end_time"],
        "test_start_time": info["test_start_time"],
        "train_points": info["train_points"],
        "val_points": info["val_points"],
        "test_points": info["test_points"],
    }
    for segment_name, info in split_info.items()
])

display(split_df)

split_path = os.path.join(REPORTS_DIR, "22_split_resumen.csv")
split_df.to_csv(split_path, index=False)
print("Split guardado en:", split_path)

In [ ]:
fig, axes = plt.subplots(len(segment_names), 1, figsize=(16, 4 * len(segment_names)), sharex=False)
if len(segment_names) == 1:
    axes = [axes]

for ax, segment_name in zip(axes, segment_names):
    df_segment = segments[segment_name]
    train_end_time = split_info[segment_name]["train_end_time"]
    test_start_time = split_info[segment_name]["test_start_time"]

    train_df = df_segment.loc[df_segment.index < train_end_time]
    val_df = df_segment.loc[(df_segment.index >= train_end_time) & (df_segment.index < test_start_time)]
    test_df = df_segment.loc[df_segment.index >= test_start_time]

    ax.plot(train_df.index, train_df[TARGET_COL], label="Entrenamiento", color="tab:blue", linewidth=0.8)
    ax.plot(val_df.index, val_df[TARGET_COL], label="Validacion", color="tab:orange", linewidth=0.8)
    ax.plot(test_df.index, test_df[TARGET_COL], label="Test", color="tab:green", linewidth=0.8)

    ax.axvline(train_end_time, color="black", linestyle="--", linewidth=1, label="Fin train")
    ax.axvline(test_start_time, color="black", linestyle=":", linewidth=1.2, label="Inicio test")

    ax.set_title(f"Split entrenamiento-validacion-test - segmento {segment_name}")
    ax.set_ylabel("Nivel")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper left")

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "01_split_train_val_test_por_segmento.png"), dpi=180, bbox_inches="tight")
plt.show()

## Escalado y ventanas para LSTM

## De series temporales a ejemplos supervisados

Una LSTM no se entrena directamente con una columna de tiempo completa. Primero hay que transformar la serie temporal en un problema supervisado. Para cada origen de prediccion se toma una ventana historica y se define un objetivo futuro.

Si `input_length = 72`, cada ejemplo contiene 72 pasos de 10 minutos, es decir 12 horas de historia. Como usamos dos variables por paso (`Nivel` y `Precipitacion`), cada entrada tiene forma:

```text
X_i: matriz de 72 x 2
```

donde las dos columnas son:

```text
columna 0: nivel escalado
columna 1: precipitacion escalada
```

El objetivo asociado tiene forma:

```text
y_i: vector de 3 valores
```

correspondientes a:

```text
[Nivel(t+10 min), Nivel(t+20 min), Nivel(t+30 min)]
```

Por tanto, el arreglo completo que entra al modelo tiene forma:

```text
X_train: [numero_de_ventanas, input_length, numero_de_features]
y_train: [numero_de_ventanas, forecast_horizon]
```

En este notebook:

```text
numero_de_features = 2
forecast_horizon = 3
```

Cuando el grid search prueba `input_length = 72`, `144` o `288`, lo que cambia es el numero de pasos historicos observados por la LSTM:

- 72 pasos = 12 horas;
- 144 pasos = 24 horas;
- 288 pasos = 48 horas.

Esto permite evaluar si el sistema hidrologico se predice mejor con memoria corta, diaria o mas extendida.

## Escalado: por que es necesario y como se hace

Las redes neuronales son sensibles a la escala numerica de las variables. Si una variable tiene magnitudes mucho mayores que otra, puede dominar los gradientes y dificultar el entrenamiento. Por eso aqui se escala tanto el nivel como la precipitacion.

El escalado se ajusta **solo con el tramo de entrenamiento** de cada segmento. Esto evita que estadisticas de validation o test entren indirectamente al modelo. Para cada segmento se calculan:

```text
nivel_min_train, nivel_max_train
lluvia_min_train, lluvia_max_train
```

y luego se transforma:

```text
valor_escalado = (valor - minimo_train) / (maximo_train - minimo_train)
```

Hay un detalle importante: el escalado se calcula por segmento, no globalmente. Esto permite que las ventanas de 2021 y 2025 queden en una escala comparable para la red, sin forzar que ambos periodos tengan exactamente la misma distribucion absoluta. Al predecir, el modelo entrega valores escalados; despues se aplica la transformacion inversa correspondiente al segmento para volver a unidades originales de nivel.

Este procedimiento tambien respeta la separacion temporal: el escalador de cada segmento no mira los datos de test.

In [ ]:
def safe_scale(min_value, max_value):
    scale = float(max_value - min_value)
    if not np.isfinite(scale) or abs(scale) < 1e-12:
        return 1.0
    return scale


scalers = {}
scaled_arrays = {}

for segment_name in segment_names:
    df_segment = segments[segment_name]
    train_end_position = split_info[segment_name]["train_end_position"]
    train_df = df_segment.iloc[:train_end_position]

    target_min = float(train_df[TARGET_COL].min())
    target_max = float(train_df[TARGET_COL].max())
    rain_min = float(train_df[RAIN_COL].min())
    rain_max = float(train_df[RAIN_COL].max())

    target_scale = safe_scale(target_min, target_max)
    rain_scale = safe_scale(rain_min, rain_max)

    scalers[segment_name] = {
        "target_min": target_min,
        "target_scale": target_scale,
        "rain_min": rain_min,
        "rain_scale": rain_scale,
    }

    target_scaled = ((df_segment[TARGET_COL].astype(float).values - target_min) / target_scale).astype(np.float32)
    rain_scaled = ((df_segment[RAIN_COL].astype(float).values - rain_min) / rain_scale).astype(np.float32)

    scaled_arrays[segment_name] = {
        "target": target_scaled,
        "rain": rain_scaled,
        "features": np.column_stack([target_scaled, rain_scaled]).astype(np.float32),
    }


def inverse_target(segment_name, values_scaled):
    scaler = scalers[segment_name]
    return np.asarray(values_scaled, dtype=float) * scaler["target_scale"] + scaler["target_min"]


def build_window_arrays(segment_name, start_origin, end_origin, input_length):
    features = scaled_arrays[segment_name]["features"]
    target = scaled_arrays[segment_name]["target"]

    X_rows = []
    y_rows = []
    origins = []

    input_length = int(input_length)
    start_origin = max(int(start_origin), input_length)
    end_origin = int(end_origin)

    if end_origin < start_origin:
        return (
            np.empty((0, input_length, features.shape[1]), dtype=np.float32),
            np.empty((0, FORECAST_HORIZON), dtype=np.float32),
            np.empty((0,), dtype=int),
        )

    for origin in range(start_origin, end_origin + 1):
        X_rows.append(features[origin - input_length:origin])
        y_rows.append(target[origin:origin + FORECAST_HORIZON])
        origins.append(origin)

    return np.stack(X_rows), np.stack(y_rows), np.asarray(origins, dtype=int)


def build_train_val_arrays(input_length):
    train_parts = []
    val_parts = []
    window_summary_rows = []

    for segment_name in segment_names:
        info = split_info[segment_name]

        X_train_seg, y_train_seg, train_origins = build_window_arrays(
            segment_name,
            start_origin=input_length,
            end_origin=info["train_end_position"] - FORECAST_HORIZON,
            input_length=input_length,
        )
        X_val_seg, y_val_seg, val_origins = build_window_arrays(
            segment_name,
            start_origin=info["train_end_position"],
            end_origin=info["test_start_position"] - FORECAST_HORIZON,
            input_length=input_length,
        )

        train_parts.append((X_train_seg, y_train_seg))
        val_parts.append((X_val_seg, y_val_seg))

        window_summary_rows.append({
            "segmento": segment_name,
            "input_length": int(input_length),
            "train_windows": len(X_train_seg),
            "validation_windows": len(X_val_seg),
            "primer_origen_train": train_origins[0] if len(train_origins) else np.nan,
            "ultimo_origen_train": train_origins[-1] if len(train_origins) else np.nan,
            "primer_origen_validation": val_origins[0] if len(val_origins) else np.nan,
            "ultimo_origen_validation": val_origins[-1] if len(val_origins) else np.nan,
        })

    X_train_local = np.concatenate([part[0] for part in train_parts], axis=0)
    y_train_local = np.concatenate([part[1] for part in train_parts], axis=0)
    X_val_local = np.concatenate([part[0] for part in val_parts], axis=0)
    y_val_local = np.concatenate([part[1] for part in val_parts], axis=0)
    summary_local = pd.DataFrame(window_summary_rows)

    return X_train_local, y_train_local, X_val_local, y_val_local, summary_local


X_train, y_train, X_val, y_val, window_summary_df = build_train_val_arrays(LSTM_INPUT_LENGTH)

display(window_summary_df)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

## Modelo LSTM

## Arquitectura usada

La arquitectura implementada es deliberadamente simple para que sea comparable y controlable:

```text
Entrada X: [batch, input_length, 2]
        |
        v
LSTM: procesa la secuencia paso a paso
        |
        v
ultimo hidden state: resumen aprendido de la ventana historica
        |
        v
Dropout: regularizacion
        |
        v
Capa lineal: produce 3 valores futuros
        |
        v
Salida y_hat: [batch, 3]
```

La LSTM procesa toda la ventana historica, pero para predecir usamos el ultimo estado oculto. Ese estado funciona como una representacion comprimida de la dinamica reciente del sistema. Luego una capa lineal transforma esa representacion en tres valores: nivel a 10, 20 y 30 minutos.

### Por que no se usa lluvia futura?

Porque en un sistema de alerta temprana normalmente no se conoce con certeza la lluvia futura inmediata. Por eso el modelo solo recibe lluvia observada dentro de la ventana historica. Si en otro experimento se quisiera usar pronostico de lluvia, eso deberia tratarse como otra configuracion metodologica y compararse por separado.

### Funcion de perdida

El entrenamiento minimiza MSE sobre la escala normalizada:

```text
MSE = promedio((y_real_escalado - y_predicho_escalado)^2)
```

El uso de MSE penaliza mas los errores grandes, lo cual puede ser util para eventos de crecida, pero tambien puede hacer que el modelo sea sensible a picos atipicos. Por eso las metricas finales se reportan en unidades originales y se complementan con MAE, RMSE, R2, MAPE, sMAPE, Bias y NSE.

## Grid search de hiperparametros

El grid search replica la logica del notebook 09: se define una lista de configuraciones, se entrena una corrida por configuracion, se evalua en validation y se selecciona la mejor segun RMSE global.

En este notebook se optimizan:

- `input_length`: longitud de la ventana historica;
- `hidden_size`: dimension del estado oculto de la LSTM;
- `num_layers`: numero de capas LSTM apiladas;
- `dropout`: regularizacion entre capas y antes de la salida;
- `learning_rate`: paso del optimizador Adam;
- `batch_size`: cantidad de ventanas procesadas por actualizacion;
- `n_epochs`: cantidad maxima de pasadas sobre el entrenamiento.

La grilla mantiene `forecast_horizon = 3` porque el objetivo cientifico es comparar contra Ridge-ARX, N-BEATS, ARIMA y SARIMAX en los mismos horizontes de 10, 20 y 30 minutos.

La seleccion de hiperparametros usa validation. El test no participa en esta etapa. Despues de elegir la mejor configuracion, se entrena el modelo final y recien entonces se calcula test.

In [ ]:
def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class LSTMForecaster(nn.Module):
    def __init__(self, n_features, hidden_size, num_layers, dropout, horizon):
        super().__init__()
        recurrent_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=recurrent_dropout,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        output, _ = self.lstm(x)
        last_hidden = output[:, -1, :]
        return self.head(self.dropout(last_hidden))


def make_loader(X, y, batch_size, shuffle, pin_memory=False):
    dataset = TensorDataset(
        torch.from_numpy(X).float(),
        torch.from_numpy(y).float(),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=pin_memory)


def evaluate_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_n = 0
    with torch.no_grad():
        for xb, yb in loader:
            non_blocking = device.type == "cuda"
            xb = xb.to(device, non_blocking=non_blocking)
            yb = yb.to(device, non_blocking=non_blocking)
            pred = model(xb)
            loss = criterion(pred, yb)
            batch_n = len(xb)
            total_loss += float(loss.item()) * batch_n
            total_n += batch_n
    return total_loss / max(total_n, 1)


set_global_seed(RANDOM_STATE)

try:
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("CUDA device:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    torch.backends.cudnn.benchmark = True
else:
    print("CUDA no disponible en este entorno; el notebook queda preparado para usarla si existe.")

pin_memory = device.type == "cuda"

EPS = 1e-8


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0:
        return {
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "R2": np.nan,
            "MAPE (%)": np.nan,
            "sMAPE (%)": np.nan,
            "Bias": np.nan,
            "NSE": np.nan,
        }

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan

    valid_mape = np.abs(y_true) > EPS
    mape = np.mean(np.abs((y_true[valid_mape] - y_pred[valid_mape]) / y_true[valid_mape])) * 100 if valid_mape.any() else np.nan

    smape_denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    valid_smape = smape_denom > EPS
    smape = np.mean(np.abs(y_true[valid_smape] - y_pred[valid_smape]) / smape_denom[valid_smape]) * 100 if valid_smape.any() else np.nan

    bias = np.mean(y_pred - y_true)
    denominator = np.sum((y_true - np.mean(y_true)) ** 2)
    nse = 1 - np.sum((y_true - y_pred) ** 2) / denominator if denominator > EPS else np.nan

    return {
        "n": len(y_true),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE (%)": mape,
        "sMAPE (%)": smape,
        "Bias": bias,
        "NSE": nse,
    }


def predict_numpy(model, X, batch_size, device):
    model.eval()
    outputs = []
    local_pin_memory = device.type == "cuda"
    loader = DataLoader(
        torch.from_numpy(X).float(),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=local_pin_memory,
    )
    with torch.no_grad():
        for xb in loader:
            xb = xb.to(device, non_blocking=local_pin_memory)
            pred = model(xb).detach().cpu().numpy()
            outputs.append(pred)
    if not outputs:
        return np.empty((0, FORECAST_HORIZON), dtype=np.float32)
    return np.concatenate(outputs, axis=0)


def context_for_segment(segment_name):
    df_segment = segments[segment_name]
    context = df_segment[[TARGET_COL, RAIN_COL]].copy()
    context["lluvia_1h"] = context[RAIN_COL].rolling(STEPS_PER_HOUR, min_periods=1).sum()
    context["delta_nivel"] = context[TARGET_COL].diff()
    context = context.rename(columns={TARGET_COL: "actual"})
    context["pred_time"] = context.index
    return context


def enrich_prediction_rows(segment_name, model_name, rows):
    pred_df = pd.DataFrame(rows)
    if pred_df.empty:
        return pred_df

    context = context_for_segment(segment_name)

    pred_df["segmento"] = segment_name
    pred_df["modelo"] = model_name
    pred_df = pred_df.merge(
        context[["pred_time", "actual", RAIN_COL, "lluvia_1h", "delta_nivel"]],
        on="pred_time",
        how="left",
    )

    return pred_df.dropna(subset=["actual", "pred"]).reset_index(drop=True)


def train_lstm_once(params, run_id):
    input_length = int(params["input_length"])
    batch_size = int(params["batch_size"])
    n_epochs = int(params["n_epochs"])

    X_train_local, y_train_local, X_val_local, y_val_local, summary_local = build_train_val_arrays(input_length)

    set_global_seed(RANDOM_STATE)

    model_local = LSTMForecaster(
        n_features=X_train_local.shape[-1],
        hidden_size=int(params["hidden_size"]),
        num_layers=int(params["num_layers"]),
        dropout=float(params["dropout"]),
        horizon=FORECAST_HORIZON,
    ).to(device)

    train_loader_local = make_loader(X_train_local, y_train_local, batch_size, shuffle=True, pin_memory=pin_memory)
    val_loader_local = make_loader(X_val_local, y_val_local, batch_size, shuffle=False, pin_memory=pin_memory)

    criterion_local = nn.MSELoss()
    optimizer_local = torch.optim.Adam(
        model_local.parameters(),
        lr=float(params["learning_rate"]),
        weight_decay=float(params["weight_decay"]),
    )

    history_local = []
    best_state = None
    best_val_loss = np.inf
    best_epoch = None

    start_time = time.time()

    for epoch in range(1, n_epochs + 1):
        model_local.train()
        train_loss_total = 0.0
        train_n = 0

        for xb, yb in train_loader_local:
            xb = xb.to(device, non_blocking=pin_memory)
            yb = yb.to(device, non_blocking=pin_memory)

            optimizer_local.zero_grad(set_to_none=True)
            pred = model_local(xb)
            loss = criterion_local(pred, yb)
            loss.backward()
            optimizer_local.step()

            batch_n = len(xb)
            train_loss_total += float(loss.item()) * batch_n
            train_n += batch_n

        train_loss = train_loss_total / max(train_n, 1)
        val_loss = evaluate_loss(model_local, val_loader_local, criterion_local, device)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model_local.state_dict())

        history_local.append({
            "run_id": run_id,
            "epoch": epoch,
            "train_loss_mse_scaled": train_loss,
            "val_loss_mse_scaled": val_loss,
            "best_epoch_so_far": best_epoch,
            "best_val_loss_mse_scaled": best_val_loss,
        })

        print(
            f"{run_id} | epoch {epoch:02d}/{n_epochs} - "
            f"train_loss={train_loss:.6f} - val_loss={val_loss:.6f} - best_epoch={best_epoch}"
        )

    elapsed = time.time() - start_time

    if best_state is not None:
        model_local.load_state_dict(best_state)

    history_local_df = pd.DataFrame(history_local)
    history_local_df["elapsed_seconds_total"] = elapsed

    train_info = {
        "elapsed_seconds": elapsed,
        "best_epoch": best_epoch,
        "best_val_loss_mse_scaled": best_val_loss,
        "train_loss_final": history_local_df["train_loss_mse_scaled"].iloc[-1] if len(history_local_df) else np.nan,
        "val_loss_final": history_local_df["val_loss_mse_scaled"].iloc[-1] if len(history_local_df) else np.nan,
        "X_train_shape": str(tuple(X_train_local.shape)),
        "X_val_shape": str(tuple(X_val_local.shape)),
    }

    return model_local, history_local_df, summary_local, train_info


def make_lstm_prediction_range(segment_name, model_for_eval, input_length, batch_size, start_origin, end_origin, model_name):
    df_segment = segments[segment_name]
    time_index = df_segment.index

    X_eval_seg, _, eval_origins = build_window_arrays(
        segment_name,
        start_origin=start_origin,
        end_origin=end_origin,
        input_length=input_length,
    )
    pred_scaled = predict_numpy(model_for_eval, X_eval_seg, batch_size, device)
    pred_original = inverse_target(segment_name, pred_scaled)

    rows = []
    for origin, forecast in zip(eval_origins, pred_original):
        origin_time = time_index[origin - 1]
        for lead_step in range(1, FORECAST_HORIZON + 1):
            pred_index = origin + lead_step - 1
            rows.append({
                "origin_time": origin_time,
                "pred_time": time_index[pred_index],
                "lead_step": lead_step,
                "lead_minutes": lead_step * 10,
                "pred": float(forecast[lead_step - 1]),
            })

    return enrich_prediction_rows(segment_name, model_name, rows)


def evaluate_validation_predictions(model_for_eval, params, run_id):
    validation_predictions = {}

    for segment_name in segment_names:
        info = split_info[segment_name]
        pred_df = make_lstm_prediction_range(
            segment_name=segment_name,
            model_for_eval=model_for_eval,
            input_length=int(params["input_length"]),
            batch_size=int(params["batch_size"]),
            start_origin=info["train_end_position"],
            end_origin=info["test_start_position"] - FORECAST_HORIZON,
            model_name=run_id,
        )
        validation_predictions[(run_id, segment_name)] = pred_df

    return validation_predictions


def validation_metrics_rows(validation_predictions, params, run_id, train_info):
    rows = []
    global_parts = []

    for segment_name in segment_names:
        eval_df = validation_predictions[(run_id, segment_name)]
        metrics = compute_metrics(eval_df["actual"].values, eval_df["pred"].values)
        metrics["run_id"] = run_id
        metrics["segmento"] = segment_name
        rows.append(metrics)
        global_parts.append(eval_df)

    global_df = pd.concat(global_parts, axis=0)
    metrics = compute_metrics(global_df["actual"].values, global_df["pred"].values)
    metrics["run_id"] = run_id
    metrics["segmento"] = "global"
    rows.append(metrics)

    metrics_df_run = pd.DataFrame(rows)

    for key, value in params.items():
        metrics_df_run[key] = value
    for key, value in train_info.items():
        metrics_df_run[key] = value

    ordered_cols = [
        "run_id",
        "segmento",
        "input_length",
        "forecast_horizon",
        "hidden_size",
        "num_layers",
        "dropout",
        "learning_rate",
        "weight_decay",
        "n_epochs",
        "batch_size",
        "n",
        "MAE",
        "RMSE",
        "R2",
        "MAPE (%)",
        "sMAPE (%)",
        "Bias",
        "NSE",
        "train_loss_final",
        "val_loss_final",
        "best_epoch",
        "best_val_loss_mse_scaled",
        "elapsed_seconds",
        "X_train_shape",
        "X_val_shape",
    ]
    return metrics_df_run[ordered_cols]


def run_lstm_grid_search():
    metrics_rows = []
    run_rows = []

    for run_index, params in enumerate(LSTM_GRID, start=1):
        run_id = (
            f"run_{run_index:03d}"
            f"_in{params['input_length']}"
            f"_h{params['forecast_horizon']}"
            f"_hs{params['hidden_size']}"
            f"_l{params['num_layers']}"
            f"_do{str(params['dropout']).replace('.', 'p')}"
            f"_lr{str(params['learning_rate']).replace('.', 'p')}"
            f"_ep{params['n_epochs']}"
            f"_bs{params['batch_size']}"
        )

        print("=" * 100)
        print("Iniciando:", run_id)
        print(params)

        status = "ok"
        error_message = ""
        train_info = {}

        try:
            model_candidate, history_candidate, _, train_info = train_lstm_once(params, run_id)
            validation_predictions = evaluate_validation_predictions(model_candidate, params, run_id)
            metrics_run = validation_metrics_rows(validation_predictions, params, run_id, train_info)
            metrics_rows.append(metrics_run)

            global_row = metrics_run[metrics_run["segmento"] == "global"].iloc[0]
            validation_rmse_global = float(global_row["RMSE"])
            validation_mae_global = float(global_row["MAE"])

        except Exception as exc:
            status = "error"
            error_message = str(exc)
            validation_rmse_global = np.nan
            validation_mae_global = np.nan
            print("ERROR:", error_message)

        run_row = {
            "run_id": run_id,
            "status": status,
            "error_message": error_message,
            "validation_RMSE_global": validation_rmse_global,
            "validation_MAE_global": validation_mae_global,
            **params,
            **train_info,
        }
        run_rows.append(run_row)

        if "model_candidate" in locals():
            del model_candidate
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print("Finalizado:", run_id, "| status:", status)

    metrics_grid_df = pd.concat(metrics_rows, axis=0, ignore_index=True) if metrics_rows else pd.DataFrame()
    runs_grid_df = pd.DataFrame(run_rows)

    metrics_grid_path = os.path.join(REPORTS_DIR, "22_lstm_gridsearch_metricas.csv")
    runs_grid_path = os.path.join(REPORTS_DIR, "22_lstm_gridsearch_resumen_corridas.csv")
    metrics_grid_df.to_csv(metrics_grid_path, index=False)
    runs_grid_df.to_csv(runs_grid_path, index=False)

    print("Metricas grid search guardadas en:", metrics_grid_path)
    print("Resumen de corridas guardado en:", runs_grid_path)

    if metrics_grid_df.empty:
        raise RuntimeError("El grid search no produjo metricas validas.")

    global_grid = metrics_grid_df[metrics_grid_df["segmento"] == "global"].copy()
    global_grid = global_grid.sort_values(["RMSE", "MAE", "run_id"]).reset_index(drop=True)
    best_row = global_grid.iloc[0]

    best_params = {
        "input_length": int(best_row["input_length"]),
        "forecast_horizon": int(best_row["forecast_horizon"]),
        "hidden_size": int(best_row["hidden_size"]),
        "num_layers": int(best_row["num_layers"]),
        "dropout": float(best_row["dropout"]),
        "learning_rate": float(best_row["learning_rate"]),
        "weight_decay": float(best_row["weight_decay"]),
        "n_epochs": int(best_row["n_epochs"]),
        "batch_size": int(best_row["batch_size"]),
    }

    best_config_df = pd.DataFrame([{
        "selected_run_id": best_row["run_id"],
        **best_params,
        "validation_n": int(best_row["n"]),
        "validation_MAE": float(best_row["MAE"]),
        "validation_RMSE": float(best_row["RMSE"]),
        "validation_R2": float(best_row["R2"]),
        "validation_NSE": float(best_row["NSE"]),
        "validation_Bias": float(best_row["Bias"]),
    }])
    best_config_path = os.path.join(REPORTS_DIR, "22_lstm_gridsearch_mejor_config.csv")
    best_config_df.to_csv(best_config_path, index=False)

    display(global_grid.head(20))
    display(best_config_df)
    print("Mejor configuracion guardada en:", best_config_path)

    return best_params, str(best_row["run_id"]), metrics_grid_df, runs_grid_df, best_config_df


if RUN_LSTM_GRID_SEARCH:
    best_lstm_params, selected_grid_run_id, lstm_grid_metrics_df, lstm_grid_runs_df, best_lstm_config_df = run_lstm_grid_search()
else:
    selected_grid_run_id = "manual_config"
    best_lstm_params = {
        "input_length": LSTM_INPUT_LENGTH,
        "forecast_horizon": FORECAST_HORIZON,
        "hidden_size": LSTM_HIDDEN_SIZE,
        "num_layers": LSTM_NUM_LAYERS,
        "dropout": LSTM_DROPOUT,
        "learning_rate": LSTM_LEARNING_RATE,
        "weight_decay": LSTM_WEIGHT_DECAY,
        "n_epochs": LSTM_EPOCHS,
        "batch_size": LSTM_BATCH_SIZE,
    }
    best_lstm_config_df = pd.DataFrame([{"selected_run_id": selected_grid_run_id, **best_lstm_params}])

LSTM_INPUT_LENGTH = int(best_lstm_params["input_length"])
LSTM_EPOCHS = int(best_lstm_params["n_epochs"])
LSTM_BATCH_SIZE = int(best_lstm_params["batch_size"])
LSTM_HIDDEN_SIZE = int(best_lstm_params["hidden_size"])
LSTM_NUM_LAYERS = int(best_lstm_params["num_layers"])
LSTM_DROPOUT = float(best_lstm_params["dropout"])
LSTM_LEARNING_RATE = float(best_lstm_params["learning_rate"])
LSTM_WEIGHT_DECAY = float(best_lstm_params["weight_decay"])
LSTM_MODEL_NAME = f"22_lstm_vs_modelos_{selected_grid_run_id}_final"

X_train, y_train, X_val, y_val, window_summary_df = build_train_val_arrays(LSTM_INPUT_LENGTH)

print("Configuracion LSTM seleccionada para entrenamiento final:")
display(best_lstm_config_df)
display(window_summary_df)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

model = LSTMForecaster(
    n_features=X_train.shape[-1],
    hidden_size=LSTM_HIDDEN_SIZE,
    num_layers=LSTM_NUM_LAYERS,
    dropout=LSTM_DROPOUT,
    horizon=FORECAST_HORIZON,
).to(device)

train_loader = make_loader(X_train, y_train, LSTM_BATCH_SIZE, shuffle=True, pin_memory=pin_memory)
val_loader = make_loader(X_val, y_val, LSTM_BATCH_SIZE, shuffle=False, pin_memory=pin_memory)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LSTM_LEARNING_RATE,
    weight_decay=LSTM_WEIGHT_DECAY,
)

history = []
best_state = None
best_val_loss = np.inf
best_epoch = None

start_time = time.time()

for epoch in range(1, LSTM_EPOCHS + 1):
    model.train()
    train_loss_total = 0.0
    train_n = 0

    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=pin_memory)
        yb = yb.to(device, non_blocking=pin_memory)

        optimizer.zero_grad(set_to_none=True)
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        batch_n = len(xb)
        train_loss_total += float(loss.item()) * batch_n
        train_n += batch_n

    train_loss = train_loss_total / max(train_n, 1)
    val_loss = evaluate_loss(model, val_loader, criterion, device)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())

    history.append({
        "epoch": epoch,
        "train_loss_mse_scaled": train_loss,
        "val_loss_mse_scaled": val_loss,
        "best_epoch_so_far": best_epoch,
        "best_val_loss_mse_scaled": best_val_loss,
    })

    print(
        f"Epoch {epoch:02d}/{LSTM_EPOCHS} - "
        f"train_loss={train_loss:.6f} - val_loss={val_loss:.6f} - best_epoch={best_epoch}"
    )

elapsed = time.time() - start_time
print(f"LSTM entrenado en {elapsed:.1f} segundos")

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_df["elapsed_seconds_total"] = elapsed
display(history_df.tail())

history_path = os.path.join(REPORTS_DIR, "22_lstm_entrenamiento.csv")
history_df.to_csv(history_path, index=False)
print("Historial guardado en:", history_path)

config_df = pd.DataFrame([{
    "modelo": "LSTM",
    "selected_grid_run_id": selected_grid_run_id,
    "input_length": LSTM_INPUT_LENGTH,
    "forecast_horizon": FORECAST_HORIZON,
    "epochs": LSTM_EPOCHS,
    "batch_size": LSTM_BATCH_SIZE,
    "hidden_size": LSTM_HIDDEN_SIZE,
    "num_layers": LSTM_NUM_LAYERS,
    "dropout": LSTM_DROPOUT,
    "learning_rate": LSTM_LEARNING_RATE,
    "weight_decay": LSTM_WEIGHT_DECAY,
    "random_state": RANDOM_STATE,
    "best_epoch": best_epoch,
    "best_val_loss_mse_scaled": best_val_loss,
    "grid_validation_RMSE": float(best_lstm_config_df["validation_RMSE"].iloc[0]) if "validation_RMSE" in best_lstm_config_df.columns else np.nan,
    "grid_validation_MAE": float(best_lstm_config_df["validation_MAE"].iloc[0]) if "validation_MAE" in best_lstm_config_df.columns else np.nan,
    "grid_validation_NSE": float(best_lstm_config_df["validation_NSE"].iloc[0]) if "validation_NSE" in best_lstm_config_df.columns else np.nan,
    "device": str(device),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
    "elapsed_seconds": elapsed,
}])
config_path = os.path.join(REPORTS_DIR, "22_lstm_configuracion.csv")
config_df.to_csv(config_path, index=False)
display(config_df)
print("Configuracion guardada en:", config_path)

checkpoint_path = os.path.join(MODELS_DIR, LSTM_MODEL_NAME + ".pt")
torch.save(
    {
        "state_dict": model.state_dict(),
        "config": config_df.iloc[0].to_dict(),
        "scalers": scalers,
        "split_info": split_info,
        "feature_columns": [TARGET_COL, RAIN_COL],
    },
    checkpoint_path,
)
print("Checkpoint guardado en:", checkpoint_path)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_df["epoch"], history_df["train_loss_mse_scaled"], label="train")
ax.plot(history_df["epoch"], history_df["val_loss_mse_scaled"], label="validation")
ax.axvline(best_epoch, color="black", linestyle="--", linewidth=1, label="best epoch")
ax.set_title("Curva de entrenamiento LSTM")
ax.set_xlabel("Epoca")
ax.set_ylabel("MSE escalado")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "02_lstm_training_loss.png"), dpi=180, bbox_inches="tight")
plt.show()

## Predicciones secuenciales sobre test

## Como se usa la LSTM entrenada para predecir

Una vez entrenada, la LSTM se usa de forma deslizante sobre el conjunto de test. Para cada origen de prediccion:

1. Se toma una ventana historica de longitud `input_length`.
2. La ventana contiene nivel y lluvia ya observados.
3. El modelo produce simultaneamente tres predicciones.
4. Cada prediccion se guarda en formato largo, con su `origin_time`, `pred_time`, `lead_step` y `lead_minutes`.

Ejemplo conceptual:

```text
origin_time = 2026-02-15 03:00

Entrada:
  datos observados desde origin_time - input_length*10min
  hasta origin_time

Salida:
  prediccion para 2026-02-15 03:10
  prediccion para 2026-02-15 03:20
  prediccion para 2026-02-15 03:30
```

El notebook usa `EVAL_STRIDE = 1`, por lo que genera predicciones en cada origen posible del test. Esto coincide con la evaluacion secuencial usada en los notebooks finales.

### Formato largo de predicciones

El formato largo es importante porque permite calcular metricas globales y tambien metricas por horizonte. Una misma ventana genera tres filas:

```text
lead_step = 1, lead_minutes = 10
lead_step = 2, lead_minutes = 20
lead_step = 3, lead_minutes = 30
```

Cada fila se enriquece con:

- `actual`: nivel observado en el instante futuro;
- `pred`: nivel predicho;
- `Precipitacion`: lluvia observada en el instante predicho, usada solo para analisis posterior;
- `lluvia_1h`: lluvia acumulada reciente asociada al instante predicho;
- `delta_nivel`: cambio observado del nivel.

Estas variables enriquecidas no se usan para entrenar la LSTM en test; se agregan despues para calcular condiciones criticas y analizar errores.

In [ ]:
predictions = {}

for segment_name in segment_names:
    info = split_info[segment_name]
    df_segment = segments[segment_name]
    time_index = df_segment.index

    X_test_seg, _, test_origins = build_window_arrays(
        segment_name,
        start_origin=info["test_start_position"],
        end_origin=len(df_segment) - FORECAST_HORIZON,
        input_length=LSTM_INPUT_LENGTH,
    )
    pred_scaled = predict_numpy(model, X_test_seg, LSTM_BATCH_SIZE, device)
    pred_original = inverse_target(segment_name, pred_scaled)

    rows = []
    for origin, forecast in zip(test_origins, pred_original):
        origin_time = time_index[origin - 1]
        for lead_step in range(1, FORECAST_HORIZON + 1):
            pred_index = origin + lead_step - 1
            rows.append({
                "origin_time": origin_time,
                "pred_time": time_index[pred_index],
                "lead_step": lead_step,
                "lead_minutes": lead_step * 10,
                "pred": float(forecast[lead_step - 1]),
            })

    pred_df = enrich_prediction_rows(segment_name, "LSTM", rows)
    predictions[("LSTM", segment_name)] = pred_df
    print(("LSTM", segment_name), len(pred_df))

predictions_long_df = pd.concat(predictions.values(), axis=0).reset_index(drop=True)
display(predictions_long_df.head())

predictions_path = os.path.join(REPORTS_DIR, "22_predicciones_test_long.csv")
predictions_long_df.to_csv(predictions_path, index=False)
print("Predicciones de test guardadas en:", predictions_path)

## Metricas

## Como interpretar las metricas

El notebook reporta metricas en unidades originales del nivel, no en escala normalizada. Esto es fundamental para comparar con Ridge-ARX, N-BEATS, ARIMA y SARIMAX.

Las metricas principales son:

- `MAE`: error absoluto medio. Es facil de interpretar y menos sensible a errores extremos.
- `RMSE`: raiz del error cuadratico medio. Penaliza con mas fuerza los errores grandes; por eso suele ser importante en crecidas.
- `R2`: proporcion de varianza explicada respecto a un modelo constante.
- `MAPE (%)`: error porcentual absoluto medio; puede ser inestable cuando el nivel real esta cerca de cero.
- `sMAPE (%)`: version simetrica del MAPE.
- `Bias`: promedio de `pred - actual`. Si es positivo, el modelo tiende a sobreestimar; si es negativo, tiende a subestimar.
- `NSE`: eficiencia de Nash-Sutcliffe, muy usada en hidrologia. Valores cercanos a 1 indican buen ajuste; valores menores que 0 indican que el promedio observado podria ser mejor que el modelo.

Se calculan tres tipos de tablas:

1. **Global y por segmento**: mezcla todos los horizontes o separa 2021/2025.
2. **Por lead time**: separa t+10, t+20 y t+30.
3. **Por condiciones criticas**: evalua si el modelo funciona en lluvia, niveles altos o ascensos rapidos.

Para alerta temprana, las metricas globales pueden ser demasiado optimistas, porque hay muchos periodos tranquilos. Por eso las tablas por condicion critica son tan importantes como la tabla global.

In [ ]:
EPS = 1e-8


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0:
        return {
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "R2": np.nan,
            "MAPE (%)": np.nan,
            "sMAPE (%)": np.nan,
            "Bias": np.nan,
            "NSE": np.nan,
        }

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    if len(np.unique(y_true)) > 1:
        r2 = r2_score(y_true, y_pred)
    else:
        r2 = np.nan

    valid_mape = np.abs(y_true) > EPS
    if valid_mape.any():
        mape = np.mean(np.abs((y_true[valid_mape] - y_pred[valid_mape]) / y_true[valid_mape])) * 100
    else:
        mape = np.nan

    smape_denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    valid_smape = smape_denom > EPS
    if valid_smape.any():
        smape = np.mean(np.abs(y_true[valid_smape] - y_pred[valid_smape]) / smape_denom[valid_smape]) * 100
    else:
        smape = np.nan

    bias = np.mean(y_pred - y_true)
    denominator = np.sum((y_true - np.mean(y_true)) ** 2)
    nse = 1 - np.sum((y_true - y_pred) ** 2) / denominator if denominator > EPS else np.nan

    return {
        "n": len(y_true),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE (%)": mape,
        "sMAPE (%)": smape,
        "Bias": bias,
        "NSE": nse,
    }


def build_metrics_table():
    rows = []
    model_names = sorted(set(model_name for model_name, _ in predictions.keys()))

    for model_name in model_names:
        global_parts = []

        for segment_name in segment_names:
            eval_df = predictions.get((model_name, segment_name))
            if eval_df is None or eval_df.empty:
                continue

            metrics = compute_metrics(eval_df["actual"].values, eval_df["pred"].values)
            metrics["modelo"] = model_name
            metrics["segmento"] = segment_name
            rows.append(metrics)
            global_parts.append(eval_df)

        if global_parts:
            global_df = pd.concat(global_parts, axis=0)
            metrics = compute_metrics(global_df["actual"].values, global_df["pred"].values)
            metrics["modelo"] = model_name
            metrics["segmento"] = "global"
            rows.append(metrics)

    cols = ["modelo", "segmento", "n", "MAE", "RMSE", "R2", "MAPE (%)", "sMAPE (%)", "Bias", "NSE"]
    return pd.DataFrame(rows)[cols].sort_values(["segmento", "RMSE", "modelo"]).reset_index(drop=True)


def build_lead_metrics_table():
    rows = []
    model_names = sorted(set(model_name for model_name, _ in predictions.keys()))

    for model_name in model_names:
        global_parts = []

        for segment_name in segment_names:
            eval_df = predictions.get((model_name, segment_name))
            if eval_df is None or eval_df.empty:
                continue

            for lead_minutes, lead_df in eval_df.groupby("lead_minutes"):
                metrics = compute_metrics(lead_df["actual"].values, lead_df["pred"].values)
                metrics["modelo"] = model_name
                metrics["segmento"] = segment_name
                metrics["lead_minutes"] = int(lead_minutes)
                rows.append(metrics)

            global_parts.append(eval_df)

        if global_parts:
            global_df = pd.concat(global_parts, axis=0)
            for lead_minutes, lead_df in global_df.groupby("lead_minutes"):
                metrics = compute_metrics(lead_df["actual"].values, lead_df["pred"].values)
                metrics["modelo"] = model_name
                metrics["segmento"] = "global"
                metrics["lead_minutes"] = int(lead_minutes)
                rows.append(metrics)

    cols = ["modelo", "segmento", "lead_minutes", "n", "MAE", "RMSE", "R2", "MAPE (%)", "sMAPE (%)", "Bias", "NSE"]
    return pd.DataFrame(rows)[cols].sort_values(["segmento", "lead_minutes", "RMSE", "modelo"]).reset_index(drop=True)


metrics_df = build_metrics_table()
lead_metrics_df = build_lead_metrics_table()

display(metrics_df)
display(lead_metrics_df)

metrics_path = os.path.join(REPORTS_DIR, "22_metricas_global_segmento.csv")
lead_metrics_path = os.path.join(REPORTS_DIR, "22_metricas_por_lead.csv")
metrics_df.to_csv(metrics_path, index=False)
lead_metrics_df.to_csv(lead_metrics_path, index=False)

print("Metricas globales/segmento guardadas en:", metrics_path)
print("Metricas por lead guardadas en:", lead_metrics_path)

## Metricas por condiciones criticas

## Por que evaluar condiciones criticas

En series hidrologicas urbanas puede ocurrir que un modelo tenga excelente desempeno promedio, pero falle justo cuando mas importa: durante ascensos rapidos o niveles altos. Para evitar esa lectura incompleta, este notebook calcula metricas en subconjuntos de interes:

- `precipitacion_gt_0`: instantes con lluvia observada;
- `lluvia_1h_alta`: instantes con lluvia acumulada reciente alta;
- `nivel_p90`: instantes donde el nivel esta en el percentil 90 o superior;
- `delta_positivo_alto`: instantes de ascenso fuerte del nivel.

Estas condiciones se calculan sobre las filas de prediccion ya generadas. Es decir, primero el modelo predice; luego se clasifica cada instante segun el contexto observado. Esto permite responder preguntas mas relevantes para gestion de riesgo:

- El modelo solo es bueno en periodos estables?
- Subestima los ascensos rapidos?
- Mejora o empeora frente a N-BEATS en lluvia fuerte?
- A t+30 minutos mantiene desempeno suficiente para alerta temprana?

Por eso las tablas por condicion y por horizonte son esenciales para una evaluacion academica y operacional.

In [ ]:
def condition_masks(eval_df):
    masks = {}
    masks["todos"] = pd.Series(True, index=eval_df.index)
    masks["precipitacion_gt_0"] = eval_df[RAIN_COL] > 0

    positive_rain_1h = eval_df.loc[eval_df["lluvia_1h"] > 0, "lluvia_1h"]
    if len(positive_rain_1h) > 0:
        rain_1h_threshold = positive_rain_1h.quantile(0.75)
        masks["lluvia_1h_alta"] = eval_df["lluvia_1h"] >= rain_1h_threshold
    else:
        masks["lluvia_1h_alta"] = pd.Series(False, index=eval_df.index)

    level_threshold = eval_df["actual"].quantile(0.90)
    masks["nivel_p90"] = eval_df["actual"] >= level_threshold

    positive_delta = eval_df.loc[eval_df["delta_nivel"] > 0, "delta_nivel"]
    if len(positive_delta) > 0:
        delta_threshold = positive_delta.quantile(0.90)
        masks["delta_positivo_alto"] = eval_df["delta_nivel"] >= delta_threshold
    else:
        masks["delta_positivo_alto"] = pd.Series(False, index=eval_df.index)

    return masks


def build_condition_metrics_tables():
    condition_rows = []
    condition_lead_rows = []
    model_names = sorted(set(model_name for model_name, _ in predictions.keys()))

    for model_name in model_names:
        global_parts = []

        for segment_name in segment_names:
            eval_df = predictions.get((model_name, segment_name))
            if eval_df is None or eval_df.empty:
                continue

            global_parts.append(eval_df)
            masks = condition_masks(eval_df)

            for condition_name, mask in masks.items():
                subset = eval_df.loc[mask]
                metrics = compute_metrics(subset["actual"].values, subset["pred"].values)
                metrics["modelo"] = model_name
                metrics["segmento"] = segment_name
                metrics["condicion"] = condition_name
                condition_rows.append(metrics)

                for lead_minutes, lead_df in subset.groupby("lead_minutes"):
                    lead_metrics = compute_metrics(lead_df["actual"].values, lead_df["pred"].values)
                    lead_metrics["modelo"] = model_name
                    lead_metrics["segmento"] = segment_name
                    lead_metrics["condicion"] = condition_name
                    lead_metrics["lead_minutes"] = int(lead_minutes)
                    condition_lead_rows.append(lead_metrics)

        if global_parts:
            global_df = pd.concat(global_parts, axis=0).reset_index(drop=True)
            masks = condition_masks(global_df)

            for condition_name, mask in masks.items():
                subset = global_df.loc[mask]
                metrics = compute_metrics(subset["actual"].values, subset["pred"].values)
                metrics["modelo"] = model_name
                metrics["segmento"] = "global"
                metrics["condicion"] = condition_name
                condition_rows.append(metrics)

                for lead_minutes, lead_df in subset.groupby("lead_minutes"):
                    lead_metrics = compute_metrics(lead_df["actual"].values, lead_df["pred"].values)
                    lead_metrics["modelo"] = model_name
                    lead_metrics["segmento"] = "global"
                    lead_metrics["condicion"] = condition_name
                    lead_metrics["lead_minutes"] = int(lead_minutes)
                    condition_lead_rows.append(lead_metrics)

    condition_metrics_df = pd.DataFrame(condition_rows)
    condition_metrics_df = condition_metrics_df[
        ["modelo", "segmento", "condicion", "n", "MAE", "RMSE", "R2", "MAPE (%)", "sMAPE (%)", "Bias", "NSE"]
    ].sort_values(["condicion", "segmento", "RMSE", "modelo"]).reset_index(drop=True)

    condition_lead_metrics_df = pd.DataFrame(condition_lead_rows)
    condition_lead_metrics_df = condition_lead_metrics_df[
        ["modelo", "segmento", "condicion", "lead_minutes", "n", "MAE", "RMSE", "R2", "MAPE (%)", "sMAPE (%)", "Bias", "NSE"]
    ].sort_values(["condicion", "segmento", "lead_minutes", "RMSE", "modelo"]).reset_index(drop=True)

    return condition_metrics_df, condition_lead_metrics_df


condition_metrics_df, condition_lead_metrics_df = build_condition_metrics_tables()

display(condition_metrics_df)
display(condition_lead_metrics_df)

condition_metrics_path = os.path.join(REPORTS_DIR, "22_metricas_condiciones.csv")
condition_lead_metrics_path = os.path.join(REPORTS_DIR, "22_metricas_condiciones_por_lead.csv")

condition_metrics_df.to_csv(condition_metrics_path, index=False)
condition_lead_metrics_df.to_csv(condition_lead_metrics_path, index=False)

print("Metricas por condicion guardadas en:", condition_metrics_path)
print("Metricas por condicion y lead guardadas en:", condition_lead_metrics_path)

## Guia practica para usar o modificar este notebook

Si queres hacer una corrida exploratoria rapida, reduci temporalmente `LSTM_GRID` a pocas configuraciones. Por ejemplo, deja solo una ventana historica y un `hidden_size`. Cuando confirmes que todo corre bien en CUDA, podes volver a activar la grilla completa.

Parametros que conviene tocar primero:

- `input_length`: controla cuanta historia ve el modelo. Es el hiperparametro mas directamente ligado a la memoria hidrologica.
- `hidden_size`: controla la capacidad interna de la LSTM. Valores muy altos pueden sobreajustar.
- `num_layers`: apilar capas permite representaciones mas complejas, pero aumenta costo y riesgo de sobreajuste.
- `dropout`: ayuda a regularizar, especialmente con varias capas.
- `learning_rate`: si el entrenamiento es inestable, bajarlo suele ayudar.
- `n_epochs`: si validation sigue mejorando, puede aumentarse; si validation empeora, hay sobreajuste.

Senales de alerta durante el entrenamiento:

- train loss baja pero validation loss sube: posible sobreajuste;
- validation RMSE muy diferente entre 2021 y 2025: posible falta de generalizacion;
- buen RMSE global pero mal `delta_positivo_alto`: el modelo no sirve bien para crecidas;
- bias negativo en eventos: el modelo subestima niveles altos o ascensos.

En la PC con GPU, la primera celda de entrenamiento debe imprimir `Device: cuda`. Si imprime `Device: cpu`, PyTorch no esta usando CUDA aunque haya tarjeta grafica instalada.

## Salidas generadas

El notebook deja las tablas LSTM y las tablas comparativas en:

- `reports/22_lstm_vs_modelos/22_metricas_global_segmento.csv`
- `reports/22_lstm_vs_modelos/22_metricas_por_lead.csv`
- `reports/22_lstm_vs_modelos/22_metricas_condiciones.csv`
- `reports/22_lstm_vs_modelos/22_metricas_condiciones_por_lead.csv`
- `reports/22_lstm_vs_modelos/22_lstm_gridsearch_metricas.csv`
- `reports/22_lstm_vs_modelos/22_lstm_gridsearch_resumen_corridas.csv`
- `reports/22_lstm_vs_modelos/22_lstm_gridsearch_mejor_config.csv`


El checkpoint entrenado queda en `models/22_lstm_vs_modelos`.